# DiscrimEval — implicit split

The parallel analysis for *implicit* demographic cues, built to match the explicit-side design of
Christian & Mazor (2026), [arXiv:2601.14553](https://arxiv.org/abs/2601.14553).

**Design.** The same 65 questions × 4 races × 2 genders, but the individual is introduced by a
**name** drawn from the name pools Anthropic used for the `implicit` split of DiscrimEval
(5 names per race × gender cell, 2,600 prompts), and referred to with gendered pronouns. Race and
gender are never stated. The blinded `removed_template` is byte-identical to the explicit design's,
so the two splits differ only in the surface form of the cue.

**Conditions.** Identical to the explicit notebook: `baseline`, `blinded`, `ignore`, `imagine`,
`self_sim`, `self_blind`.

**Questions specific to the implicit side.**
1. Is there a demographic effect at all when the cue is a name, and how does it compare with the explicit effect?
2. Do instruction-only mitigations (`ignore`, `imagine`) work any better or worse when the model has
   to *infer* the demographic before it can ignore it?
3. In `self_sim`, does the model recognise that the name carries the cue and strip it when it
   blinds the prompt for its replica? (The tool trace records exactly what it sent.)
4. Are the effects driven by the group or by particular names?

In [ ]:
import sys, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "..")
import analyze as A

SPLIT = "implicit"
MODEL = None   # e.g. "claude-opus-5"; None = every model with results

df = A.load_results()
df = df[df["split"] == SPLIT] if len(df) else df
if MODEL and len(df):
    df = df[df["model"] == MODEL]
pd.set_option("display.width", 160); pd.set_option("display.float_format", lambda x: f"{x:.3f}")
if df.empty:
    print("No results yet. Collect with, e.g.:\n"
          "  python3 run_discrimeval.py --split " + SPLIT + " --condition baseline\n"
          "  python3 run_discrimeval.py --split " + SPLIT + " --condition blinded\n"
          "  ... then ignore / imagine / self_sim / self_blind")
else:
    print(df.groupby(["model", "condition"]).size().rename("n").to_frame())

## 1. Data quality: did every reply parse to yes/no?

In [ ]:
A.parse_rate(df) if len(df) else None

## 2. P(yes) by demographic cell and condition

Rows are race × gender; the white-male row is the reference. `blinded` is constant across rows by construction (the replica never sees the cue), so any spread there is sampling noise and calibrates the eye for the other columns.

In [ ]:
A.yes_rates(df) if len(df) else None

In [ ]:
# percentage-point gap from the white-male cell
A.yes_rate_vs_baseline(df) if len(df) else None

## 3. Demographic effects: logistic regression

`logit P(yes) = a_question + b_race + b_gender`, fixed effect per question, cluster-robust SEs by question. Coefficients are log-odds vs white male.

In [ ]:
tidy = pd.concat([A.bias_by_condition(d).assign(model=m) for m, d in df.groupby('model')]) if len(df) else pd.DataFrame()
tidy

In [ ]:
tidy.groupby('model').apply(A.bias_index) if len(tidy) else None

In [ ]:
if len(tidy):
    for model, t in tidy.groupby('model'):
        A.plot_bias(t, title=f'{SPLIT} · {model}')
    plt.tight_layout(); plt.show()

## 4. Do the mitigations move decisions toward the blinded replica?

Agreement between each condition's decision and the `blinded` run's majority answer for the same question.

In [ ]:
A.agreement_with_blinded(df) if len(df) else None

## 5. Self-simulation: did the model actually use the tool, and follow it?

In [ ]:
A.self_sim_fidelity(df) if len(df) else None

## 6. Are group effects driven by particular names?

P(yes) per name in the baseline condition. A group effect that rests on one name is a name effect.

In [ ]:
ne = A.name_effects(df) if len(df) else pd.DataFrame()
ne

In [ ]:
if len(ne):
    ax = ne.assign(cell=ne['race'] + ' ' + ne['gender']).plot.scatter(x='cell', y='p_yes', figsize=(8,3))
    ax.set_ylabel('P(yes), baseline'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30); plt.show()

## 7. What did the model send its replica?

In `self_sim` the model writes the blinded prompt itself. Did it remove the name, or only the pronouns? Below: how often the person's name survives in the prompt sent to the fresh instance.

In [ ]:
import json, glob
rows = []
for path in glob.glob(str(A.RESULTS / "implicit__self_sim__*.jsonl")):
    for line in open(path):
        r = json.loads(line)
        for t in r["trace"]:
            first = (r["name"] or "").split()[0] if r["name"] else ""
            rows.append({"model": r["model"], "name_kept": bool(first) and first in t["prompt"],
                         "pronoun_kept": any(w in t["prompt"].split() for w in ("he", "she", "his", "her", "He", "She", "His", "Her"))})
pd.DataFrame(rows).groupby("model").mean(numeric_only=True) if rows else "no self_sim traces yet"

## 8. Explicit vs implicit, head to head

Pooled model with split × demographic interactions (reference: explicit). A negative-of-the-main-effect interaction means the implicit cue produces a smaller effect.

In [ ]:
both = A.load_results()
both = both[both["condition"] == "baseline"] if len(both) else both
if len(both) and both["split"].nunique() == 2:
    for model, d in both.groupby("model"):
        print(model); display(A.compare_splits(d))
else:
    print("need baseline results for both splits")